# NuFold 10-target pilot batch runner

Runs the repository's frozen pilot manifest. Inputs and results are stored on mounted Google Drive. Native structures are used only after inference for evaluation. Select a GPU runtime before running. Set `MSA_MODE` to `genuine` only when the Drive MSA folder contains a query-matched A3M for every target. Resume by reusing `RUN_ID` and setting `RESUME = True`.

For a private repository, the preferred route is a Colab Secret named `GITHUB_TOKEN` containing a fine-grained, read-only Contents token. Enable notebook access to that secret. The notebook never prints the token or puts it in the clone URL/command. Without the secret, it prompts for a ZIP downloaded while signed into GitHub (or already in Drive). The frozen pilot manifest is intentionally not committed as a dataset; provide the existing CSV from Drive or upload it when prompted.

In [ ]:
from google.colab import drive, userdata, files
from pathlib import Path
from datetime import datetime, timezone
import os, shutil, subprocess, sys, tempfile, zipfile
drive.mount('/content/drive')
REPO = Path('/content/rna3d-template-sparse-benchmark')
REPO_URL = 'https://github.com/ABDULSALAM589/rna3d-template-sparse-benchmark.git'
REQUIRED_CODE = ['scripts/nufold/run_batch.py', 'scripts/nufold/evaluate.py', 'scripts/nufold/common.py',
                 'scripts/nufold/evaluate_run.py', 'configs/nufold_pilot.json',
                 'tests/test_nufold_pipeline.py']
MANIFEST_REL = Path('data/pilot/rna3d_template_sparse_pilot_v1.csv')
def repo_has_code(root):
    return root.is_dir() and all((root / name).is_file() for name in REQUIRED_CODE)
def repo_is_complete(root):
    return repo_has_code(root) and (root / MANIFEST_REL).is_file()
def install_source_zip():
    zip_path = Path('/content/rna3d-template-sparse-benchmark.zip')
    drive_zip = Path('/content/drive/MyDrive/RNA3D_Benchmark/rna3d-template-sparse-benchmark.zip')
    if drive_zip.is_file():
        shutil.copy2(drive_zip, zip_path)
    elif not zip_path.is_file():
        print('Upload the repository ZIP downloaded from GitHub while signed in.')
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError('No source ZIP was uploaded.')
        name, payload = next(iter(uploaded.items()))
        if not name.lower().endswith('.zip'):
            raise RuntimeError('Upload a .zip archive of this repository.')
        zip_path.write_bytes(payload)
    staging = Path('/content/nufold-source-unpacked')
    shutil.rmtree(staging, ignore_errors=True)
    with zipfile.ZipFile(zip_path) as archive:
        members = archive.infolist()
        if any(Path(member.filename).is_absolute() or '..' in Path(member.filename).parts for member in members):
            raise RuntimeError('Unsafe path found in source ZIP.')
        archive.extractall(staging)
    candidates = [p.parent for p in staging.rglob('scripts/nufold/run_batch.py')]
    root = next((p for p in candidates if repo_has_code(p)), None)
    if root is None:
        raise RuntimeError('ZIP is missing required NuFold pipeline files or frozen pilot manifest.')
    if REPO.exists(): shutil.rmtree(REPO)
    shutil.copytree(root, REPO)
    print('Loaded repository source from ZIP.')
token = None
try:
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    pass
if REPO.exists() and not repo_is_complete(REPO):
    raise RuntimeError(f'Existing repository folder is incomplete: {REPO}')
if not REPO.exists() and token:
    with tempfile.NamedTemporaryFile('w', prefix='gh-askpass-', dir='/content', delete=False) as helper:
        helper.write("#!/usr/bin/env python3\nimport os, sys\nprompt = sys.argv[1].lower() if len(sys.argv) > 1 else ''\nprint('x-access-token' if 'username' in prompt else os.environ['GH_REPO_TOKEN'])\n")
        askpass = Path(helper.name)
    askpass.chmod(0o700)
    clone_env = os.environ.copy()
    clone_env.update({'GIT_ASKPASS': str(askpass), 'GIT_TERMINAL_PROMPT': '0', 'GH_REPO_TOKEN': token})
    try:
        result = subprocess.run(['git', 'clone', '--quiet', REPO_URL, str(REPO)], env=clone_env,
                                stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=False)
        if result.returncode != 0:
            shutil.rmtree(REPO, ignore_errors=True)
            print('Authenticated clone was unavailable; switching to the ZIP workflow.')
    finally:
        clone_env.pop('GH_REPO_TOKEN', None)
        token = None
        askpass.unlink(missing_ok=True)
if not REPO.exists():
    install_source_zip()
manifest_path = REPO / MANIFEST_REL
if not manifest_path.is_file():
    drive_manifest = Path('/content/drive/MyDrive/RNA3D_Benchmark/rna3d_template_sparse_pilot_v1.csv')
    if drive_manifest.is_file():
        manifest_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(drive_manifest, manifest_path)
    else:
        print('Upload the existing frozen pilot CSV (not a new or edited manifest).')
        uploaded = files.upload()
        match = next(((name, payload) for name, payload in uploaded.items() if name.endswith('.csv')), None)
        if match is None: raise RuntimeError('No pilot manifest CSV was uploaded.')
        manifest_path.parent.mkdir(parents=True, exist_ok=True)
        manifest_path.write_bytes(match[1])
if not repo_is_complete(REPO):
    raise RuntimeError('Repository source is missing required pipeline files.')
print('Pipeline and frozen pilot manifest verified before dependency installation:', REPO)

In [ ]:
# Install the same NuFold dependencies used by the repository's verified notebook.
subprocess.run([sys.executable,'-m','pip','install','-q','ml-collections','dm-tree','deepspeed','protobuf','scipy','biopython','numpy','matplotlib','py3Dmol'],check=True)
NUFOLD = Path('/content/NuFold')
if not NUFOLD.exists():
    subprocess.run(['git','clone','https://github.com/kiharalab/NuFold.git',str(NUFOLD)],check=True)
subprocess.run(['git','-C',str(NUFOLD),'checkout','f77d34af97c1d0f88b47dd2f3c038004fe2ebe13'],check=True)
CKPT = NUFOLD/'checkpoints/global_step145245.pt'
CKPT.parent.mkdir(parents=True,exist_ok=True)
if not CKPT.exists():
    subprocess.run(['wget','-O',str(CKPT),'http://kiharalab.org/nufold/global_step145245.pt'],check=True)
assert CKPT.is_file() and CKPT.stat().st_size > 0
print('NuFold commit:',subprocess.check_output(['git','-C',str(NUFOLD),'rev-parse','HEAD'],text=True).strip())
print('Checkpoint bytes:',CKPT.stat().st_size)

In [ ]:
# IPknot 1.1.0 (same release used by the existing NuFold Colab notebook).
IPKNOT = Path('/content/ipknot')
if not IPKNOT.exists():
    z = Path('/content/ipknot-1.1.0-x86_64-linux.zip')
    subprocess.run(['wget','-q','-O',str(z),'https://github.com/satoken/ipknot/releases/download/v1.1.0/ipknot-1.1.0-x86_64-linux.zip'],check=True)
    subprocess.run(['unzip','-oq',str(z),'-d','/content/ipknot-dist'],check=True)
    src = Path('/content/ipknot-dist/ipknot-1.1.0-x86_64-linux/ipknot')
    subprocess.run(['cp',str(src),str(IPKNOT)],check=True)
    IPKNOT.chmod(0o755)
# USalign pinned to official release tag 20260527.
USALIGN_ROOT = Path('/content/USalign')
if not USALIGN_ROOT.exists():
    subprocess.run(['git','clone','--depth','1','--branch','20260527','https://github.com/pylelab/USalign.git',str(USALIGN_ROOT)],check=True)
subprocess.run(['make'],cwd=USALIGN_ROOT,check=True)
USALIGN = Path('/usr/local/bin/USalign')
subprocess.run(['cp',str(USALIGN_ROOT/'USalign'),str(USALIGN)],check=True)
USALIGN.chmod(0o755)
print('IPknot:',IPKNOT,'USalign:',USALIGN)

In [ ]:
# Fail before starting the batch if this runtime is CPU-only.
import torch
print('PyTorch:',torch.__version__,'CUDA:',torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Select a Colab GPU runtime before running NuFold.')
print('GPU:',torch.cuda.get_device_name(0))
subprocess.run([sys.executable,'-m','unittest','discover','-s',str(REPO/'tests'),'-v'],cwd=REPO,check=True)

## Configure and run

Default is the clearly labeled single-sequence baseline. For genuine MSA mode, set `MSA_MODE = 'genuine'`, point `MSA_SOURCE` to the Drive folder containing target A3Ms, and provide provenance. To resume, copy the printed run ID back into `RUN_ID` and set `RESUME = True`.

In [ ]:
RUN_ROOT = Path('/content/drive/MyDrive/RNA3D_Benchmark/NuFold/runs')
MSA_MODE = 'single_sequence'  # or 'genuine'
MSA_SOURCE = Path('/content/drive/MyDrive/RNA3D_Benchmark/MSA/pilot_v1')
MSA_PROVENANCE = ''  # required for genuine mode: DB/release/tool/date/settings
RUN_ID = 'pilot10_single_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
RESUME = False
cmd = [sys.executable,str(REPO/'scripts/nufold/run_batch.py'),'--config',str(REPO/'configs/nufold_pilot.json'),
       '--run-id',RUN_ID,'--msa-mode',MSA_MODE,'--run-root',str(RUN_ROOT),
       '--nufold-root',str(NUFOLD),'--checkpoint',str(CKPT),'--ipknot',str(IPKNOT),
       '--usalign',str(USALIGN)]
if MSA_MODE == 'genuine':
    if not MSA_PROVENANCE.strip(): raise ValueError('Set MSA_PROVENANCE before genuine MSA mode.')
    cmd += ['--msa-source-dir',str(MSA_SOURCE),'--msa-provenance',MSA_PROVENANCE]
if RESUME: cmd.append('--resume')
print('Run ID:',RUN_ID)
print('Command:', ' '.join(cmd))
subprocess.run(cmd,cwd=REPO,check=True)